# 1. Preprocessing

In [8]:
import pandas as pd
import numpy as np 
import sklearn
import networkx as nx

import ast

In [2]:
train = pd.read_csv("./data/train.csv")
test = pd.read_csv("./data/test.csv")

In [4]:
print(train.head())

   language  sentence   n                                           edgelist  \
0  Japanese         2  23  [(6, 4), (2, 6), (2, 23), (20, 2), (15, 20), (...   
1  Japanese         5  18  [(8, 9), (14, 8), (4, 14), (5, 4), (1, 2), (6,...   
2  Japanese         8  33  [(2, 10), (2, 14), (4, 2), (16, 4), (6, 16), (...   
3  Japanese        11  30  [(30, 1), (14, 24), (21, 14), (3, 21), (7, 3),...   
4  Japanese        12  19  [(19, 13), (16, 19), (2, 16), (4, 10), (4, 15)...   

   root  
0    10  
1    10  
2     3  
3    30  
4    11  


## Binary Classification Setup 

### Feature Engineering

In [ ]:
def get_graph_features(G, node, root):
    """Extract rich graph features for a given node in tree G."""
    features = {}

    # Degree-based (4 features)
    features['in_degree'] = G.in_degree(node)
    features['out_degree'] = G.out_degree(node)
    features['degree'] = G.degree(node)
    features['is_leaf'] = int(G.out_degree(node) == 0)

    # Distance from root (1 feature)
    # FOR NOW LETS NOT USE IT SINCE IT CAN PRODUCE NULL VALUES 
    # (ALTHOUGH IN THEORY THIS ARE TREES SO THEY ARE CONNECTED SOMEHOW)
    #try:
    #    features['depth_from_root'] = nx.shortest_path_length(G, source=root, target=node)
    #except nx.NetworkXNoPath:
    #    features['depth_from_root'] = None

    # Subtree size (1 features)
    features['descendants'] = len(nx.descendants(G, node))

    # Centralities (5 features)
    # Compute all at once for efficiency
    closeness = nx.closeness_centrality(G)
    betweenness = nx.betweenness_centrality(G)
    pagerank = nx.pagerank(G, alpha=0.85)
    harmonic = nx.harmonic_centrality(G)
    try:
        eigen = nx.eigenvector_centrality_numpy(G)
    except nx.NetworkXException:
        eigen = {n: 0.0 for n in G.nodes()}

    features['closeness'] = closeness.get(node, 0)
    features['betweenness'] = betweenness.get(node, 0)
    features['pagerank'] = pagerank.get(node, 0)
    features['harmonic'] = harmonic.get(node, 0)
    features['eigenvector'] = eigen.get(node, 0)

    # AVOID FOR NOW SINCE IT CAN CAUSE NULL VALUES
    # Eccentricity and average path length (2 features)
    #try:
    #    features['eccentricity'] = nx.eccentricity(G)[node]
    #except:
    #    features['eccentricity'] = None

    #try:
    #    lengths = nx.single_source_shortest_path_length(G, node)
    #    features['avg_shortest_path_from_node'] = sum(lengths.values()) / len(lengths)
    #except:
    #    features['avg_shortest_path_from_node'] = None

    # in total this can create up to 12 new features
    return features


### Graph Flatten and Processing

In [ ]:
def flatten_and_add_graph_features(df):
    """
    Processes a dependency-tree dataframe into node-level format with graph features.
    Works on both train and test DataFrames.
    """
    rows = []

    for idx, row in df.iterrows():
        lang = row['language']
        sent_id = row['sentence']
        n = row['n']
        edges_raw = row['edgelist']
        root = row['root']

        # Convert string to list of tuples
        if isinstance(edges_raw, str):
            edges = ast.literal_eval(edges_raw)
        else:
            edges = edges_raw  # Already a list
        G = nx.DiGraph(edges)

        for node in G.nodes():
            node_data = {
                'language': lang,
                'sentence_id': sent_id,
                'n': n,
                'vertex_id': node,
                'is_root': int(node == root)
            }
            node_data.update(get_graph_features(G, node, root))
            rows.append(node_data)

    return pd.DataFrame(rows)


binary_train = flatten_and_add_graph_features(train)
binary_test = flatten_and_add_graph_features(test)

### Load Processed Data

In [ ]:
binary_train.to_csv("./data/processed_binary_train.csv")
binary_test.to_csv("./data/processed_binary_test.csv")